# Cat vs Dog Image Classifier on Azure ML
Run in Azure ML Studio → Notebooks, on a Compute Instance (GPU recommended, e.g. `Standard_NC6s_v3`), kernel **Python 3.10 - SDK v2**.

Expected folder layout (no pre-made split; the notebook splits train/val for you):
```
data/
  cats-dogs-small/
    cats/*.jpg
    dogs/*.jpg
```

## 1. Install dependencies

In [1]:
!pip install -q azure-ai-ml azure-identity mlflow azureml-mlflow torch torchvision scikit-learn matplotlib pillow

## 2. Connect to your Azure ML workspace

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

try:
    # Works automatically on an Azure ML compute instance
    ml_client = MLClient.from_config(credential=DefaultAzureCredential())
except Exception:
    ml_client = MLClient(
        DefaultAzureCredential(),
        subscription_id="<SUBSCRIPTION_ID>",
        resource_group_name="<RESOURCE_GROUP>",
        workspace_name="<WORKSPACE_NAME>",
    )
print(ml_client.workspace_name)

## 3. Register the dataset in Azure ML (versioned, reusable)
Upload your `data/` folder to the notebook file system first (drag and drop in Studio), or point to a blob/datastore path.

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

data_asset = Data(
    path="./data/cats-dogs-small",
    type=AssetTypes.URI_FOLDER,
    description="Cat vs Dog images (train/val)",
    name="cats-dogs-images",
)
data_asset = ml_client.data.create_or_update(data_asset)
print(data_asset.name, data_asset.version)

## 4. Load data and split into train/validation

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset

DATA_DIR = "./data/cats-dogs-small"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# Two views of the same folder: one with augmentation, one without
full_train = datasets.ImageFolder(DATA_DIR, train_tf)
full_val = datasets.ImageFolder(DATA_DIR, val_tf)

# Reproducible 80/20 split
g = torch.Generator().manual_seed(42)
perm = torch.randperm(len(full_train), generator=g).tolist()
n_train = int(0.8 * len(perm))
train_ds = Subset(full_train, perm[:n_train])
val_ds = Subset(full_val, perm[n_train:])

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=4)
val_dl = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=4)

class_names = full_train.classes   # ['cats', 'dogs']
print(class_names, "train:", len(train_ds), "val:", len(val_ds))

## 5. Build model (transfer learning with ResNet18)

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for p in model.parameters():
    p.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

## 6. Train (metrics are tracked in Azure ML via MLflow)

In [ ]:
import mlflow, copy
EPOCHS = 8
best_acc, best_wts = 0.0, None

with mlflow.start_run(run_name="resnet18-cats-dogs"):
    mlflow.log_params({"epochs": EPOCHS, "lr": 1e-3, "model": "resnet18", "batch_size": 32})
    for epoch in range(EPOCHS):
        model.train()
        run_loss = 0.0
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            run_loss += loss.item() * x.size(0)
        train_loss = run_loss / len(train_ds)

        model.eval()
        correct = 0
        with torch.no_grad():
            for x, y in val_dl:
                x, y = x.to(device), y.to(device)
                correct += (model(x).argmax(1) == y).sum().item()
        val_acc = correct / len(val_ds)

        mlflow.log_metrics({"train_loss": train_loss, "val_acc": val_acc}, step=epoch)
        print(f"Epoch {epoch+1}/{EPOCHS}  loss={train_loss:.4f}  val_acc={val_acc:.4f}")
        if val_acc > best_acc:
            best_acc, best_wts = val_acc, copy.deepcopy(model.state_dict())

    model.load_state_dict(best_wts)
    mlflow.log_metric("best_val_acc", best_acc)
print("Best val acc:", best_acc)

## 7. Evaluate

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x, y in val_dl:
        preds = model(x.to(device)).argmax(1).cpu()
        y_true += y.tolist(); y_pred += preds.tolist()

print(classification_report(y_true, y_pred, target_names=class_names))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=class_names).plot()
plt.show()

## 8. Save and register the model

In [ ]:
import os
from azure.ai.ml.entities import Model

os.makedirs("outputs", exist_ok=True)
torch.save({"state_dict": model.state_dict(), "classes": class_names}, "outputs/cats_dogs_resnet18.pt")

registered = ml_client.models.create_or_update(Model(
    path="outputs/cats_dogs_resnet18.pt",
    name="cats-dogs-resnet18",
    type="custom_model",
    description="ResNet18 transfer learning, cat vs dog",
))
print(registered.name, registered.version)

## 9. Test on a single image

In [ ]:
from PIL import Image

def predict(path):
    img = val_tf(Image.open(path).convert("RGB")).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(img), 1)[0]
    i = probs.argmax().item()
    return class_names[i], probs[i].item()

print(predict("./data/cats-dogs-small/cats/example.jpg"))  # change path

## 10. (Optional) Run as a scheduled/remote job on a GPU cluster
Move the training code into `train.py` and submit with `azure.ai.ml.command(...)` to a compute cluster so it scales and auto-shuts down.

In [ ]:
from azure.ai.ml.entities import AmlCompute
cluster = AmlCompute(name="gpu-cluster", size="Standard_NC6s_v3", min_instances=0, max_instances=2, idle_time_before_scale_down=120)
ml_client.compute.begin_create_or_update(cluster).result()